# MFN Quickstart

Morphology-aware Field Intelligence Engine — deterministic morphology intelligence for reaction-diffusion systems with causal verification.

This notebook demonstrates the full pipeline: simulate → extract → detect → explain → validate.

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mycelium_fractal_net as mfn
from mycelium_fractal_net.core.causal_validation import validate_causal_consistency

print(f"MFN v{mfn.__version__}")

## 2. Data — Three Regimes

We simulate three different dynamical regimes by varying the diffusion coefficient α:

| Regime | α | What happens |
|--------|------|-------------|
| Stable | 0.10 | Low diffusion, localized patterns, minimal morphogenesis |
| Transitional | 0.16 | Moderate diffusion, emerging Turing patterns |
| Critical | 0.22 | Near CFL limit, strong pattern formation, boundary effects |

In [ ]:
specs = {
    "stable": mfn.SimulationSpec(grid_size=32, steps=24, alpha=0.10, seed=1),
    "transitional": mfn.SimulationSpec(grid_size=32, steps=24, alpha=0.16, seed=2),
    "critical": mfn.SimulationSpec(grid_size=32, steps=24, alpha=0.22, seed=3),
}

sequences = {name: mfn.simulate(spec) for name, spec in specs.items()}

for name, seq in sequences.items():
    print(repr(seq))

## 3. Analysis

For each regime: extract morphology descriptor, detect anomalies, validate causally.

In [ ]:
results = {}

for name, seq in sequences.items():
    descriptor = seq.extract()
    anomaly = seq.detect()
    validation = validate_causal_consistency(seq, descriptor, anomaly)

    results[name] = {
        "seq": seq,
        "descriptor": descriptor,
        "anomaly": anomaly,
        "validation": validation,
    }

    print(f"{name:14s} | {repr(anomaly):60s} | causal: {validation.decision.value}")

## 4. Visualization

Field heatmaps with morphology metrics overlaid.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.patch.set_facecolor("#0a0a0a")

for col, (name, data) in enumerate(results.items()):
    seq = data["seq"]
    desc = data["descriptor"]
    anom = data["anomaly"]

    # Top row: field heatmap
    ax = axes[0, col]
    im = ax.imshow(seq.field * 1000, cmap="viridis", aspect="equal")
    ax.set_title(f"{name}\nα = {specs[name].alpha}", color="white", fontsize=11)
    ax.tick_params(colors="#666")
    plt.colorbar(im, ax=ax, label="mV", shrink=0.8)

    # Bottom row: metrics
    ax = axes[1, col]
    ax.set_facecolor("#0a0a0a")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    d_box = desc.features.get("D_box", 0)
    lzc = desc.complexity.get("temporal_lzc", 0)
    ii = desc.stability.get("instability_index", 0)

    lines = [
        f"D_box = {d_box:.3f}",
        f"LZC = {lzc:.3f}",
        f"Instability = {ii:.3f}",
        f"",
        f"Anomaly: {anom.label}",
        f"Score: {anom.score:.3f}",
        f"Regime: {anom.regime.label}",
        f"Confidence: {anom.confidence:.2f}",
    ]

    label_colors = {"nominal": "#00c896", "watch": "#f0ad4e", "anomalous": "#d9534f"}

    for i, line in enumerate(lines):
        color = "#cccccc"
        if line.startswith("Anomaly:"):
            color = label_colors.get(anom.label, "#cccccc")
        ax.text(
            0.05,
            0.92 - i * 0.12,
            line,
            transform=ax.transAxes,
            fontsize=10,
            color=color,
            fontfamily="monospace",
        )

plt.tight_layout()
plt.savefig("regime_comparison.png", dpi=150, facecolor="#0a0a0a")
plt.show()
print("Saved: regime_comparison.png")

## 5. Causal Validation Manifest

Every conclusion is verified by causal rules. Here's the breakdown per stage.

In [ ]:
# Pick the critical regime — most interesting for causal analysis
v = results["critical"]["validation"]

print(f"Decision: {v.decision.value}")
print(f"Rules: {len(v.rule_results)} checked")
print(f"Errors: {v.error_count} | Warnings: {v.warning_count}")
print()

# Group by stage
stages = {}
for r in v.rule_results:
    stages.setdefault(r.stage, []).append(r)

for stage, rules in stages.items():
    passed = sum(1 for r in rules if r.passed)
    total = len(rules)
    status = "✓" if passed == total else "⚠"
    print(f"  {status} {stage:15s} {passed}/{total}")
    for r in rules:
        if not r.passed:
            print(f"      ✗ [{r.rule_id}] {r.message}")
        elif r.spec:
            print(f"      ✓ [{r.rule_id}] {r.spec.claim[:60]}")

## 6. Decision Explainability

The system explains *why* it made each decision — not post-hoc, but as part of the computation.

In [ ]:
# Explain the critical regime
seq_critical = sequences["critical"]
explanation = seq_critical.explain()
print(explanation.narrate())

## 7. Bring Your Own Data

Load external field data through the adapter:

In [ ]:
from mycelium_fractal_net.adapters import FieldAdapter

# Option 1: from numpy array
# your_data = np.load("your_file.npy")  # shape (H,W) or (T,H,W)
# seq = FieldAdapter.load(your_data)

# Option 2: from CSV
# seq = FieldAdapter.load("measurements.csv")

# Option 3: simulate with custom parameters
seq = mfn.simulate(
    mfn.SimulationSpec(
        grid_size=64,
        steps=32,
        alpha=0.18,
        seed=42,
    )
)

# Full pipeline in 4 lines
print(repr(seq))
print(repr(seq.detect()))
print(repr(seq.extract()))
print(repr(seq.explain()))